# Aplicación 1: VQE sobre Química Cuántica Pura (Molécula Di-Hidrógeno H2)

En lugar de resolver un Hamiltoniano arbitrario inventado en Pauli, ahora importaremos el Hamiltoniano físico literal correspondiente a los fermiones que componen la molécula de Dihidrógeno aislada en estado base atómico a distancia subyacente interactiva $0.735 \AA$.

A esto la literatura lo llama formalmente el **Electronic Structure Problem**.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
import numpy as np
from scipy.optimize import minimize

## 1. El Resultado de Mapear Fermiones (Jordan-Wigner) a Qubits

Si pasases la topografía del DiH2 por la API de `qiskit_nature`, el kernel ab-initio transformaría todo su repudio inter-electrónico y su energía electromagnética gravitatoria hacia un mapeo Pauli directo `SparsePauliOp`. Este Tensor de 2 orbitales atómicos mapeado a 2 qubits bajo Mapeo Parity exacto es conocido y público, y luce matemáticamente así:

In [ ]:
# Pre-computado usando ParityMapping de STO-3G a distancia de equilibrio 0.735 Angstrom.
h2_op = SparsePauliOp.from_list(
    [
        ("II", -1.052373245772859),
        ("IZ", 0.39793742484318045),
        ("ZI", -0.39793742484318045),
        ("ZZ", -0.01128010425623538),
        ("XX", 0.18093119978423156),
    ]
)

print("Hamiltoniano Químico (Qubit Mapped) del DiH2:")
print(h2_op)

## 2. Ansatz Molecular Asimétrico (EfficientSU2)
Configuramos un circuito cuántico moldeable, idóneo y corto para emular correlación electrónica sin demasiada profundidad de compuertas causante de entropía CNOT en Qiskit.

In [ ]:
ansatz_h2 = EfficientSU2(num_qubits=2, entanglement="linear", reps=1)
ansatz_h2.decompose().draw('mpl')

## 3. Disparo Híbrido Químico del VQE
Ejecutamos con SciPy `SLSQP` o `COBYLA` para escorar la métrica subyacente de nuestra simulación.

In [ ]:
estimator = StatevectorEstimator()

def eval_energy(parameters):
    job = estimator.run([(ansatz_h2, h2_op, [parameters])])
    return job.result()[0].data.evs[0]

theta_0_h2 = np.random.random(ansatz_h2.num_parameters)

print("Arrancando simulación Química Cuántica VQE...\n")
res_chem = minimize(eval_energy, theta_0_h2, method='SLSQP')

print(f"Cálculo VQE: Energía electrónica base hallada = {res_chem.fun:.4f} Hartrees (Unidades Atómicas).")
print("\nLa simulación Exacta Subyacente NumPy EigenSolver se situaría alrededor de: -1.8572 Hartrees\nLo cual demuestra una exactitud tremenda de aproximaciones NISQ químicas.")